# Notebook 09 — Evaluation Metrics for Imbalanced Data

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 09 of 15  
**Time estimate:** 75 minutes

> Biology is almost always imbalanced. Rare diseases, rare variants, rare cell types.
> Accuracy is almost always the wrong metric. Knowing which metric to use — and why —
> is one of the most practical skills in biological ML.

---
## Step 1 — Motivation

A pathogenic variant classifier that predicts 'benign' for every variant achieves
~99.9% accuracy (most variants are benign) but has zero utility. The right metric
depends on what you care about: false positives vs. false negatives, and the
prevalence of the positive class.

---
## Step 2 — Intuition

**Confusion matrix** (binary):
```
               Predicted +    Predicted -
Actual +       TP             FN
Actual -       FP             TN
```

**Key metrics:**
- **Accuracy:** $(TP+TN)/N$ — misleading when imbalanced
- **Precision (PPV):** $TP/(TP+FP)$ — what fraction of positive predictions are correct
- **Recall (Sensitivity, TPR):** $TP/(TP+FN)$ — what fraction of true positives are found
- **F1:** $2 \times \text{Precision} \times \text{Recall} / (\text{P+R})$ — harmonic mean
- **Specificity (TNR):** $TN/(TN+FP)$
- **ROC-AUC:** Area under Receiver Operating Characteristic curve (TPR vs. FPR)
- **PR-AUC:** Area under Precision-Recall curve
- **MCC (Matthews Correlation Coefficient):** single number for all four cells

**ROC-AUC vs. PR-AUC:**
- **ROC-AUC** is insensitive to class imbalance — can look good even when
  precision is very low
- **PR-AUC** directly reflects how well you find positives without many false alarms
- For rare disease classification, variant pathogenicity, rare cell detection:
  **always report PR-AUC alongside ROC-AUC**

---
## Step 3 — Biological Background

**Pathogenic variant detection:**
- ~0.1% of all variants are pathogenic in ClinVar
- Accuracy of 99.9% (predict all benign) is useless
- Clinical reporting requires high precision: every positive report triggers expensive
  follow-up; false positives harm patients
- But recall matters: missing a pathogenic variant has consequences
- Report: AUROC, AUPRC, F1 at the operating point, and the precision-recall curve

**Rare cell type detection in scRNA-seq:**
- Rare cell types (< 1% of cells): cancer stem cells, regulatory T-cells
- Accuracy near 100% trivially; PR-AUC reveals whether you actually find them

**Drug screening:**
- 1 in 10,000 compounds is active
- Maximize recall (don't miss any active compound) at the cost of many false positives
- Report enrichment factor: fraction of actives in top-N% of ranked compounds

**SMOTE (Synthetic Minority Oversampling Technique):**
- Creates synthetic minority-class samples by interpolating between existing ones
- Better than simple oversampling (which just duplicates)
- Apply ONLY on training data, never on validation/test (another form of leakage)

**Class weighting:** Set `class_weight='balanced'` in sklearn — inversely weights
each class by its frequency. Simpler than SMOTE, often works as well.

---
## Step 4 — Mathematical Explanation

**ROC curve:** Plot TPR vs. FPR as decision threshold varies from 1 to 0.
$$\text{TPR}(t) = \frac{TP(t)}{P}, \quad \text{FPR}(t) = \frac{FP(t)}{N}$$

**AUROC interpretation:** Probability that a randomly chosen positive is ranked
higher than a randomly chosen negative by the model. 0.5 = random; 1.0 = perfect.

**PR curve:** Plot Precision vs. Recall as threshold varies.
$$\text{Precision}(t) = \frac{TP(t)}{TP(t)+FP(t)}, \quad \text{Recall}(t) = \frac{TP(t)}{P}$$

**AUPRC baseline:** For a random classifier with prevalence $\pi$: AUPRC $= \pi$.
For a 1% positive class: random classifier has AUPRC = 0.01; any useful classifier
must be much higher.

**F1 at operating threshold:**
$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**MCC:**
$$\text{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

$\text{MCC} \in [-1, 1]$; insensitive to class imbalance; preferred in many benchmarks.

In [ ]:
# Step 6 — Python: Metrics from scratch + class imbalance handling
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---- Metrics from scratch ----
def confusion_matrix_scratch(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP, FP, TN, FN

def precision_recall_f1(TP, FP, TN, FN):
    precision = TP / max(TP + FP, 1e-10)
    recall = TP / max(TP + FN, 1e-10)
    f1 = 2 * precision * recall / max(precision + recall, 1e-10)
    return precision, recall, f1

def mcc(TP, FP, TN, FN):
    denom = np.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
    return (TP*TN - FP*FN) / max(denom, 1e-10)

def roc_curve_scratch(y_true, y_scores):
    thresholds = np.sort(np.unique(y_scores))[::-1]
    tprs, fprs = [0], [0]
    P = y_true.sum()
    N = (1 - y_true).sum()
    for t in thresholds:
        y_pred = (y_scores >= t).astype(int)
        TP, FP, TN, FN = confusion_matrix_scratch(y_true, y_pred)
        tprs.append(TP / P)
        fprs.append(FP / N)
    tprs.append(1); fprs.append(1)
    auroc = np.trapz(tprs, fprs)
    return np.array(fprs), np.array(tprs), abs(auroc)

def pr_curve_scratch(y_true, y_scores):
    thresholds = np.sort(np.unique(y_scores))[::-1]
    precisions, recalls = [1], [0]
    P = y_true.sum()
    for t in thresholds:
        y_pred = (y_scores >= t).astype(int)
        TP, FP, TN, FN = confusion_matrix_scratch(y_true, y_pred)
        precisions.append(TP / max(TP+FP, 1e-10))
        recalls.append(TP / max(P, 1e-10))
    auprc = np.trapz(precisions, recalls)
    return np.array(recalls), np.array(precisions), abs(auprc)

# ---- Generate imbalanced pathogenic variant dataset ----
rng = np.random.default_rng(42)
n_total = 2000
n_path = 100  # 5% positive class (more visible than 0.1% for demo)

X_neg = rng.normal([0,0,0,0,0], 1.0, (n_total - n_path, 5))
X_pos = rng.normal([2,1,-1,0.5,1.5], 0.8, (n_path, 5))
X_imb = np.vstack([X_neg, X_pos])
y_imb = np.array([0]*(n_total-n_path) + [1]*n_path)

print(f'Dataset: {n_total} samples, {n_path} positives ({n_path/n_total:.1%})')

X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

# Compare: no class weighting vs. balanced
results = {}
for name, model in [
    ('LR (no weighting)', LogisticRegression(C=1, max_iter=1000)),
    ('LR (balanced)', LogisticRegression(C=1, class_weight='balanced', max_iter=1000)),
    ('RF (balanced)', RandomForestClassifier(100, class_weight='balanced', random_state=42)),
]:
    model.fit(X_tr_sc, y_tr)
    y_prob = model.predict_proba(X_te_sc)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    TP, FP, TN, FN = confusion_matrix_scratch(y_te, y_pred)
    prec, rec, f1 = precision_recall_f1(TP, FP, TN, FN)
    mc = mcc(TP, FP, TN, FN)
    fpr, tpr, auroc = roc_curve_scratch(y_te, y_prob)
    rec_pr, prec_pr, auprc = pr_curve_scratch(y_te, y_prob)
    results[name] = {'auroc': auroc, 'auprc': auprc, 'f1': f1, 'mcc': mc,
                       'precision': prec, 'recall': rec, 'fpr': fpr, 'tpr': tpr,
                       'rec_pr': rec_pr, 'prec_pr': prec_pr}
    print(f'{name}: AUROC={auroc:.3f}, AUPRC={auprc:.3f}, F1={f1:.3f}, MCC={mc:.3f}')

print(f'Random baseline AUPRC: {y_te.mean():.3f} (= prevalence)')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {'LR (no weighting)': 'grey', 'LR (balanced)': 'steelblue', 'RF (balanced)': 'tomato'}

# Panel A: ROC curves
ax = axes[0]
ax.plot([0,1], [0,1], 'k--', lw=0.8, label='Random')
for name, r in results.items():
    ax.plot(r['fpr'], r['tpr'], lw=2, color=colors[name],
              label=f'{name.split(" (")[0]}\nAUROC={r["auroc"]:.3f}')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('A. ROC curves')
ax.legend(fontsize=7, loc='lower right')

# Panel B: Precision-Recall curves
ax = axes[1]
ax.axhline(y_te.mean(), color='k', ls='--', lw=0.8,
             label=f'Random (AUPRC={y_te.mean():.3f})')
for name, r in results.items():
    ax.plot(r['rec_pr'], r['prec_pr'], lw=2, color=colors[name],
              label=f'{name.split(" (")[0]}\nAUPRC={r["auprc"]:.3f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('B. Precision-Recall curves\n(PR-AUC sensitive to imbalance)')
ax.legend(fontsize=7)

# Panel C: Metric comparison bar chart
ax = axes[2]
metric_names = ['AUROC', 'AUPRC', 'F1', 'MCC']
model_names = list(results.keys())
x = np.arange(len(metric_names))
width = 0.25
for i, (name, r) in enumerate(results.items()):
    vals = [r['auroc'], r['auprc'], r['f1'], r['mcc']]
    ax.bar(x + i*width, vals, width, label=name.split(' (')[0],
             color=list(colors.values())[i], alpha=0.8, edgecolor='black')
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.set_ylabel('Score')
ax.set_title('C. Metric comparison\n(balanced weighting improves F1 and MCC)')
ax.legend(fontsize=8)
ax.axhline(0.5, color='grey', ls=':', lw=0.6)

plt.suptitle('Module 13 NB09: Evaluation Metrics for Imbalanced Data', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement ROC curve computation from scratch. Verify AUROC matches
   `sklearn.metrics.roc_auc_score`.
2. Find the optimal decision threshold that maximizes F1 on the validation set.
   Show how the operating point moves on the PR curve.
3. Apply SMOTE (from `imbalanced-learn`) to the training set. Does it improve
   AUPRC compared to `class_weight='balanced'`?
4. Derive the MCC formula. Show that when the dataset is highly imbalanced and
   all predictions are negative, MCC → 0 (near-zero, not near-1).

---
## Step 10 — Quiz

1. Why is accuracy a misleading metric for imbalanced classification?
2. What is the difference between ROC-AUC and PR-AUC? Which is preferred for rare
   disease detection?
3. What is the baseline AUPRC for a random classifier on a 1%-positive dataset?
4. What does class_weight='balanced' do in sklearn?
5. Write the MCC formula. What are its range and interpretation?

---
## Step 12 — Reflection

> *[In one paragraph: explain why ROC-AUC can be misleadingly high for a model
> on a rare-event dataset, using the relationship between FPR and the number of
> false positives in the context of a 0.1%-prevalence disease.]*

---
*Next: `10_clustering_kmeans_and_hierarchical.ipynb`*